# Fine-Tuning Transformers for Arabic Sarcasm Detection

This notebook fine-tunes pre-trained Arabic language models (e.g., `aubmindlab/bert-base-arabertv02`, `asafaya/bert-base-arabic`, or `MARBERT`) on the ArSarcasm-v2 dataset.

In [1]:
import copy
import sys
import time
sys.path.append("..")

import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from src.data import get_data
from src.evaluate import log_result, metrics, plot_results, save_predictions, show
from src.preprocess import build_pipeline
from src.train import best_threshold, count_params, set_seed

MODELS = {
    "arabert": "aubmindlab/bert-base-arabertv02-twitter",
    "marbert": "UBC-NLP/MARBERTv2",
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FP16 = DEVICE == "cuda"
BATCH, MAX_LEN, LR, EPOCHS = 32, 96, 2e-5, 4
print(DEVICE)

c:\Users\rawi9\Desktop\BDIO\LLM\tps\tp2-arabic-sarcasm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cpu


In [2]:
train, val, test, report = get_data()

prep = build_pipeline(["html", "urls", "mentions"])
for frame in (train, val, test):
    frame["text"] = frame["tweet"].apply(prep)
    frame["sent"] = frame["sentiment"].str.upper().map({"NEG": 0, "NEU": 1, "POS": 2})

counts = train["sarcasm"].value_counts().sort_index()
WEIGHTS = torch.tensor(len(train) / (2 * counts.values), dtype=torch.float).to(DEVICE)

print(report)
print("class weights:", WEIGHTS.tolist())

{'start': 11293, 'in_test': 17, 'in_val': 82, 'conflicting': 102, 'duplicates': 311, 'end': 10781}
class weights: [0.603977620601654, 2.9043641090393066]
